# Prepare Data

## Pinyin Mapping

In [1]:
INITIALS = [
    "", "b","p","m","f","d","t","n","l","g","k","h",
    "j","q","x","zh","ch","sh","r","z","c","s"
]

FINALS = [
    "a","ai","an","ang","ao","e","ei","en","eng","er",
    "i","ia","ian","iang","iao","ie","in","ing","iong","iu",
    "o","ong","ou",
    "u","ua","uai","uan","uang","ui","un","uo",
    "v","ve","van","vn"
]

TONES = ["1","2","3","4","5"]

ZERO_INITIAL_MAP = {
    # y-series
    "yi":"i",
    "ya":"ia",
    "yao":"iao",
    "ye":"ie",
    "you":"iu",
    "yan":"ian",
    "yin":"in",
    "yang":"iang",
    "ying":"ing",
    "yong":"iong",

    "yu":"v",
    "yue":"ve",
    "yuan":"van",
    "yun":"vn",

    # w-series
    "wu":"u",
    "wa":"ua",
    "wo":"uo",
    "wai":"uai",
    "wei":"ui",
    "wan":"uan",
    "wen":"un",
    "wang":"uang",
    "weng":"ong",
}


def normalize_zero_initial(base):
    return ZERO_INITIAL_MAP.get(base, base)


def normalize_jqx(final):
    if final.startswith("u"):
        mapping = {
            "u": "v",
            "ue": "ve",
            "uan": "van",
            "un": "vn",
        }
        return mapping.get(final, final)
    return final


init2idx = {v:i for i,v in enumerate(INITIALS)}
final2idx = {v:i for i,v in enumerate(FINALS)}
tone2idx = {v:i for i,v in enumerate(TONES)}

idx2init = {i:v for v,i in init2idx.items()}
idx2final = {i:v for v,i in final2idx.items()}
idx2tone = {i:v for v,i in tone2idx.items()}


def decompose_pinyin(token: str):
    tone = token[-1]

    if not tone.isdigit():
        tone = "5" # neutral tone, usually has no digit
        base = token
    else:
        base = token[:-1]

    base = normalize_zero_initial(base)

    # longest-match for initials

    try:
        for ini in sorted(INITIALS, key=len, reverse=True):
            if base.startswith(ini):
                final = base[len(ini):]

                if ini in {"j","q","x"}:
                    final = normalize_jqx(final)
                
                return [init2idx[ini], final2idx[final], tone2idx[tone]]
    except Exception:
        raise ValueError("Invalid pinyin token")

## Dataset: Text + Pinyin

In [2]:
import pandas as pd

dataset = pd.read_csv("")

In [3]:
dataset.head()

,text,pinyin
0,一个特殊群体的期待在密切关注此次特金会的人中有一个特殊的群体在日本殖民期间来到日本的韩国朝鲜...,yi1 ge4 te4 shu1 qun2 ti3 de qi1 dai4 zai4 mi4...
1,他们希望政治解冻有助于让朝鲜摆脱孤立,ta1 men xi1 wang4 zheng4 zhi4 jie3 dong4 you3 ...
2,他认为对这个奖的妄想可能会促使双方做出轻率的承诺但也可能有助于艰巨的和平进程,ta1 ren4 wei2 dui4 zhe4 ge4 jiang3 de wang4 xi...
3,欢迎阅读纪思道文章的中文版,huan1 ying2 yue4 du2 ji4 si1 dao4 wen2 zhang1 ...
4,不法业者向一些中国女性编造了一个美好的未来到美国开始新生活并拥有一份合法的工作,bu4 fa3 ye4 zhe3 xiang4 yi1 xie1 zhong1 guo2 n...


## Dataset: Pinyin Embeddings

In [4]:
import pickle as pkl

with open("", "rb") as f:
    embedding_map = pkl.load(f)

In [5]:
embedding_map['wo3'].shape

(1024,)


## Training Dataset

First, filter the sequences where at least one token is missing a sound (only bo token is expected, with around 70 sentences):

In [6]:
def align(token):
    if token[-1].isalpha():  # e.g. de -> de5
        return token + '5'
    return token

# To see filtered values
dataset[
    dataset['pinyin'].str.split().transform(lambda x: any(align(y) not in embedding_map.keys() for y in x))
]

,text,pinyin
1267,纽约时报现年八十二岁的阿尔及利亚強人总统阿卜杜勒阿齐兹布特弗利卡月底将下台,niu3 yue1 shi2 bao4 xian4 nian2 ba1 shi2 er4 s...
1369,如今安倍的政治对手已经磨刀霍霍安倍经济学命运未卜,ru2 jin1 an1 bei4 de zheng4 zhi4 dui4 shou3 yi...
1440,纽约时报马尔代夫总统易卜拉欣穆罕默德萨利赫所在的政党似乎将在该国议会选举中获得压倒性胜利,niu3 yue1 shi2 bao4 ma3 er3 dai4 fu1 zong3 ton...
2203,两个家庭的命运以悲剧性的方式交汇了波尔森家失去了三个年幼的孩子炸死他们的是易卜拉欣家的两个儿子,liang3 ge4 jia1 ting2 de ming4 yun4 yi3 bei1 j...
4805,阿卜杜拉赫卜表示自己和前夫曾因此收到死亡威胁她希望公开身份能够让自己和家人免于被报复,a1 bo du4 la1 he4 bo biao3 shi4 zi4 ji3 he2 qi...
...,...,...
152505,营地在一条路边那是通往夏河县及著名的庞大藏传佛教寺院拉卜楞寺的道路,ying2 di4 zai4 yi1 tiao2 lu4 bian1 na4 shi4 to...
152511,这两位僧人曾在拉卜楞寺住过寺里的僧人赶来这里迎接他们,zhe4 liang3 wei4 seng1 ren2 ceng2 zai4 la1 bo ...
157326,如果我们收不到货东西就会更贵甚至可能要关门卖黄瓜萝卜和西红柿的左启超音说,ru2 guo3 wo3 men shou1 bu4 dao4 huo4 dong1 xi1...
157327,在他说话的同时一个女人指责他胡来提高萝卜的价格,zai4 ta1 shuo1 hua4 de tong2 shi2 yi1 ge4 nv3 ...


In [7]:
# Actual filtration
dataset = dataset[
    dataset['pinyin'].str.split().transform(lambda x: all(align(y) in embedding_map.keys() for y in x))
]

In [8]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 161392 entries, 0 to 161463
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    161392 non-null  object
 1   pinyin  161392 non-null  object
dtypes: object(2)
memory usage: 3.7+ MB


### Prepare embeddings

In [9]:
import numpy as np
import torch

all_pinyin_tokens = list(embedding_map.keys())

pinyin2idx = {tok: i for i, tok in enumerate(all_pinyin_tokens)}
embedding_matrix = torch.tensor(np.stack([embedding_map[tok] for tok in all_pinyin_tokens]), dtype=torch.float32)

### Tokenization and Batching

In [10]:
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer
from tqdm import tqdm

def prepare_bert_dataset(df, tokenizer, decompose_pinyin, pinyin2idx, align, max_length=512):
    """
    Prepares a dataset for BERT fine‑tuning on pinyin prediction.

    Args:
        df: pandas DataFrame with columns 'text' and 'pinyin'
        tokenizer: BertTokenizer
        decompose_pinyin: function (pinyin_str) -> [init_idx, final_idx, tone_idx]
        pinyin2idx: dict mapping pinyin token (e.g., 'ma1') to embedding index
        align: function to canonicalize pinyin token (e.g., remove tone digit for neutral)
        max_length: maximum sequence length after adding [CLS] and [SEP]

    Returns:
        data: dict with keys:
            'input_ids', 'attention_mask', 'pinyin_components', 'pinyin_idxs'
            each is a list of lists (per sample)
    """
    input_ids_list = []
    attention_mask_list = []
    pinyin_components_list = []
    pinyin_idxs_list = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing"):
        text = row['text']
        pinyin_str = row['pinyin']

        # Tokenize with offset mapping to align characters
        encoding = tokenizer(
            text,
            max_length=max_length,
            truncation=True,
            return_offsets_mapping=True,
            padding=False,
            return_tensors=None
        )
        # Remove offset mapping; we will use it to filter out special tokens
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        offset_mapping = encoding['offset_mapping']

        # Remove [CLS] (first token) and [SEP] (last token)
        # Also filter out any tokens that correspond to no character (offset (0,0) except specials)
        # For Chinese BERT, each character becomes one token, but we keep only those with positive offset length
        filtered_input_ids = []
        filtered_attention_mask = []
        valid_token_indices = []
        for i, (start, end) in enumerate(offset_mapping):
            if start == 0 and end == 0:
                # [CLS] or [SEP] or padding; skip if they are not at the ends? We'll just keep all? Actually we need to keep [CLS] and [SEP]? For character-level prediction we don't need them.
                # We will remove them. Since we use offset mapping, we can skip tokens that don't cover any character.
                continue
            filtered_input_ids.append(input_ids[i])
            filtered_attention_mask.append(attention_mask[i])
            valid_token_indices.append(i)

        # Check that number of remaining tokens equals number of characters
        if len(filtered_input_ids) != len(text):
            # In rare cases, a character may be split into multiple tokens; we can handle by averaging later.
            # For simplicity, skip this sample.
            print(f"Skipping sample {idx}: token count {len(filtered_input_ids)} != character count {len(text)}")
            continue

        # Pinyin tokens
        pinyin_tokens = pinyin_str.split()
        if len(pinyin_tokens) != len(text):
            print(f"Skipping sample {idx}: pinyin token count {len(pinyin_tokens)} != character count {len(text)}")
            continue

        pinyin_component_idxs = []
        pinyin_idxs = []
        skip_sample = False
        for token in pinyin_tokens:
            try:
                comp = decompose_pinyin(token)
                pinyin_component_idxs.append(comp)
                pinyin_idxs.append(pinyin2idx[align(token)])
            except Exception as e:
                print(f"Skipping sample {idx} due to pinyin error: {token} - {e}")
                skip_sample = True
                break
        if skip_sample:
            continue

        input_ids_list.append(filtered_input_ids)
        attention_mask_list.append(filtered_attention_mask)
        pinyin_components_list.append(pinyin_component_idxs)
        pinyin_idxs_list.append(pinyin_idxs)

    return {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'pinyin_components': pinyin_components_list,
        'pinyin_idxs': pinyin_idxs_list
    }

In [11]:
class BERTPinyinDataset(Dataset):
    def __init__(self, data_dict):
        self.input_ids = data_dict['input_ids']
        self.attention_mask = data_dict['attention_mask']
        self.pinyin_components = data_dict['pinyin_components']
        self.pinyin_idxs = data_dict['pinyin_idxs']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
            'pinyin_components': torch.tensor(self.pinyin_components[idx], dtype=torch.long),
            'pinyin_idxs': torch.tensor(self.pinyin_idxs[idx], dtype=torch.long)
        }

In [12]:
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
data = prepare_bert_dataset(dataset, tokenizer, decompose_pinyin, pinyin2idx, align)
dataset = BERTPinyinDataset(data)

Tokenizing: 100%|██████████| 161392/161392 [00:58<00:00, 2766.23it/s]


In [14]:
for x in dataset:
    print(x.keys())

    print(x['input_ids'].shape, x['pinyin_components'][..., 0].shape)
    break

dict_keys(['input_ids', 'attention_mask', 'pinyin_components', 'pinyin_idxs'])
torch.Size([54]) torch.Size([54])


### Dataloaders

In [15]:
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch

def collate_fn_bert(batch, pad_idx_components=-100):
    """
    Collate function for BERT pinyin dataset.
    Args:
        batch: list of dicts with keys 'input_ids', 'attention_mask', 'pinyin_components', 'pinyin_idxs'
        pad_idx_components: padding value for pinyin targets (default -100)
    Returns:
        input_ids_padded: (batch_size, max_len) LongTensor
        attention_mask_padded: (batch_size, max_len) LongTensor
        pinyin_components_padded: (batch_size, max_chars, 3) LongTensor
        pinyin_idxs_padded: (batch_size, max_chars) LongTensor
        char_lengths: (batch_size,) tensor of actual character counts (from pinyin_seq length)
    """
    input_ids = [item['input_ids'] for item in batch]
    attention_masks = [item['attention_mask'] for item in batch]
    pinyin_comp = [item['pinyin_components'] for item in batch]
    pinyin_idxs = [item['pinyin_idxs'] for item in batch]

    # Pad input_ids (pad with 0, as BERT expects)
    input_ids_padded = nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    # Pad attention_mask (pad with 0)
    attention_mask_padded = nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)
    
    # Pad pinyin components and indices (pad with pad_idx_components)
    pinyin_comp_padded = nn.utils.rnn.pad_sequence(pinyin_comp, batch_first=True, padding_value=pad_idx_components)
    pinyin_idxs_padded = nn.utils.rnn.pad_sequence(pinyin_idxs, batch_first=True, padding_value=pad_idx_components)
    
    # Character lengths per sample (number of characters)
    char_lengths = torch.tensor([len(seq) for seq in pinyin_comp], dtype=torch.long)
    
    return (input_ids_padded, attention_mask_padded, pinyin_comp_padded, pinyin_idxs_padded, char_lengths)

In [16]:
def create_bert_dataloaders(dataset, batch_size, val_split=0.1, test_split=0.1, pad_idx_components=-100, shuffle=True):
    """
    Split the BERT pinyin dataset into train, validation, test and return DataLoaders.
    Args:
        dataset: BERTPinyinDataset instance
        batch_size: int
        val_split: float (fraction for validation)
        test_split: float (fraction for test)
        pad_idx_components: int, padding value for pinyin targets
        shuffle: bool, whether to shuffle training data
    Returns:
        train_loader, val_loader, test_loader
    """
    total = len(dataset)
    val_size = int(total * val_split)
    test_size = int(total * test_split)
    train_size = total - val_size - test_size
    
    assert train_size > 0, "Not enough data for training"
    assert val_size >= 0 and test_size >= 0, "Split sizes must be non-negative"
    
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=lambda b: collate_fn_bert(b, pad_idx_components)
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda b: collate_fn_bert(b, pad_idx_components)
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda b: collate_fn_bert(b, pad_idx_components)
    )
    
    return train_loader, val_loader, test_loader

In [17]:
train_loader, val_loader, test_loader = create_bert_dataloaders(dataset, batch_size=32)

## Model Preparation

In [18]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn

# Device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
bert_model = BertModel.from_pretrained("bert-base-chinese")

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
import torch.nn as nn

class ClassifierHead(nn.Module):
    """MLP with one hidden layer for classification."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [20]:
class BertPinyinEncoder(nn.Module):
    def __init__(self, bert_model, num_initials, num_finals, num_tones, mlp_hidden_dim=256, dropout=0.3):
        super().__init__()
        self.bert = bert_model
        self.initial_head  = ClassifierHead(768, mlp_hidden_dim, num_initials, dropout)
        self.final_head    = ClassifierHead(768, mlp_hidden_dim, num_finals, dropout)
        self.tone_head     = ClassifierHead(768, mlp_hidden_dim, num_tones, dropout)
        self.phonetic_head = ClassifierHead(768, mlp_hidden_dim, 1024, dropout)

    def forward(self, input_ids, attention_mask, return_hidden=False):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state   # (batch, seq_len, 768)
        logits_i = self.initial_head(hidden)
        logits_f = self.final_head(hidden)
        logits_t = self.tone_head(hidden)

        if return_hidden:
            phonetic = self.phonetic_head(hidden)
            return logits_i, logits_f, logits_t, phonetic
        return logits_i, logits_f, logits_t

# Instantiate model
num_initials = len(init2idx)
num_finals   = len(final2idx)
num_tones    = len(tone2idx)
model = BertPinyinEncoder(bert_model, num_initials, num_finals, num_tones).to(device)

## Train Loop

In [21]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import pickle

# Loss functions
criterion_initial = nn.CrossEntropyLoss(ignore_index=-100)
criterion_final   = nn.CrossEntropyLoss(ignore_index=-100)
criterion_tone    = nn.CrossEntropyLoss(ignore_index=-100)
criterion_mse     = nn.MSELoss()

# Optimizer – lower learning rate for BERT fine‑tuning
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Training parameters
num_epochs = 20
patience = 3
lambda_mse = 0.2          # weight for MSE loss
best_val_loss = float('inf')
counter = 0

# Load embedding matrix (pre‑computed wav2vec2 embeddings) – must have dimension 768
embedding_matrix = embedding_matrix.to(device)   # shape (num_pinyin_tokens, 768)
# Normalise embeddings (optional but helps)
embedding_matrix = nn.functional.normalize(embedding_matrix, p=2, dim=1)

In [22]:
# Metrics logging
metrics_log = []

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    train_steps = 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch in train_pbar:
        input_ids, attention_mask, pinyin_comp, pinyin_idxs, char_lengths = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        pinyin_comp = pinyin_comp.to(device)          # (batch, max_chars, 3)
        pinyin_idxs = pinyin_idxs.to(device)          # (batch, max_chars)

        # Forward pass with hidden states
        logits_i, logits_f, logits_t, hidden = model(input_ids, attention_mask, return_hidden=True)
        # hidden shape: (batch, max_chars, 768)

        # Cross‑entropy losses (ignore padding)
        loss_i = criterion_initial(logits_i.reshape(-1, num_initials), pinyin_comp[..., 0].reshape(-1))
        loss_f = criterion_final(logits_f.reshape(-1, num_finals), pinyin_comp[..., 1].reshape(-1))
        loss_t = criterion_tone(logits_t.reshape(-1, num_tones), pinyin_comp[..., 2].reshape(-1))
        ce_loss = loss_i + loss_f + loss_t

        # MSE alignment loss (optional)
        mask = (pinyin_idxs != -100)   # (batch, max_chars)
        if mask.any() and lambda_mse > 0:
            valid_hidden = hidden[mask]
            valid_target_idxs = pinyin_idxs[mask]
            target_embeds = embedding_matrix[valid_target_idxs]
            # Normalise both for cosine direction
            valid_hidden_norm = nn.functional.normalize(valid_hidden, p=2, dim=-1)
            target_norm = nn.functional.normalize(target_embeds, p=2, dim=-1)
            mse_loss = criterion_mse(valid_hidden_norm, target_norm)
        else:
            mse_loss = torch.tensor(0.0, device=device)

        total_loss = mse_loss

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += total_loss.item()
        train_steps += 1
        train_pbar.set_postfix({'loss': f'{total_loss.item():.4f}', 'mse': f'{mse_loss.item():.4f}'})

    avg_train_loss = total_train_loss / train_steps

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    total_val_ce = 0
    total_val_mse = 0
    val_steps = 0
    correct_i = correct_f = correct_t = correct_exact = 0
    total_chars = 0
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for batch in val_pbar:
            input_ids, attention_mask, pinyin_comp, pinyin_idxs, char_lengths = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            pinyin_comp = pinyin_comp.to(device)
            pinyin_idxs = pinyin_idxs.to(device)

            logits_i, logits_f, logits_t, hidden = model(input_ids, attention_mask, return_hidden=True)

            loss_i = criterion_initial(logits_i.reshape(-1, num_initials), pinyin_comp[..., 0].reshape(-1))
            loss_f = criterion_final(logits_f.reshape(-1, num_finals), pinyin_comp[..., 1].reshape(-1))
            loss_t = criterion_tone(logits_t.reshape(-1, num_tones), pinyin_comp[..., 2].reshape(-1))
            ce_loss = loss_i + loss_f + loss_t

            mask = (pinyin_idxs != -100)
            if mask.any() and lambda_mse > 0:
                valid_hidden = hidden[mask]
                valid_target_idxs = pinyin_idxs[mask]
                target_embeds = embedding_matrix[valid_target_idxs]
                valid_hidden_norm = nn.functional.normalize(valid_hidden, p=2, dim=-1)
                target_norm = nn.functional.normalize(target_embeds, p=2, dim=-1)
                mse_loss = criterion_mse(valid_hidden_norm, target_norm)
            else:
                mse_loss = torch.tensor(0.0, device=device)

            total_val_ce += ce_loss.item()
            total_val_mse += mse_loss.item()
            val_loss = mse_loss
            total_val_loss += val_loss.item()
            val_steps += 1

            # Accuracy on pinyin components
            pred_i = logits_i.argmax(dim=-1)
            pred_f = logits_f.argmax(dim=-1)
            pred_t = logits_t.argmax(dim=-1)

            mask_ce = (pinyin_comp[..., 0] != -100)
            correct_i += ((pred_i == pinyin_comp[..., 0]) & mask_ce).sum().item()
            correct_f += ((pred_f == pinyin_comp[..., 1]) & mask_ce).sum().item()
            correct_t += ((pred_t == pinyin_comp[..., 2]) & mask_ce).sum().item()
            exact = (pred_i == pinyin_comp[..., 0]) & (pred_f == pinyin_comp[..., 1]) & (pred_t == pinyin_comp[..., 2]) & mask_ce
            correct_exact += exact.sum().item()
            total_chars += mask_ce.sum().item()

    avg_val_loss = total_val_loss / val_steps
    avg_val_ce = total_val_ce / val_steps
    avg_val_mse = total_val_mse / val_steps
    acc_i = correct_i / total_chars if total_chars else 0
    acc_f = correct_f / total_chars if total_chars else 0
    acc_t = correct_t / total_chars if total_chars else 0
    acc_exact = correct_exact / total_chars if total_chars else 0

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val MSE = {avg_val_mse:.4f}")
    print(f"  Init Acc: {acc_i:.4f}, Final Acc: {acc_f:.4f}, Tone Acc: {acc_t:.4f}, Exact: {acc_exact:.4f}")

    # Early stopping & save best
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'bert_pinyin_best.pt')
        print(f"  -> Best model saved (val loss: {best_val_loss:.4f})")
        counter = 0
    else:
        counter += 1
        print(f"  -> No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    scheduler.step(avg_val_loss)
    metrics_log.append({
        'epoch': epoch+1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'val_init_acc': acc_i,
        'val_final_acc': acc_f,
        'val_tone_acc': acc_t,
        'val_exact_acc': acc_exact,
        'val_mse': avg_val_mse
    })

# Save metrics
with open('bert_metrics.pkl', 'wb') as f:
    pickle.dump(metrics_log, f)
print("Training complete.")

Epoch 1/20 [Val]: 100%|██████████| 505/505 [01:05<00:00,  7.73it/s]


Epoch 1: Train Loss = 0.0004, Val Loss = 0.0002, Val MSE = 0.0002
  Init Acc: 0.0430, Final Acc: 0.0352, Tone Acc: 0.1443, Exact: 0.0000
  -> Best model saved (val loss: 0.0002)


Epoch 2/20 [Val]: 100%|██████████| 505/505 [01:05<00:00,  7.70it/s]


Epoch 2: Train Loss = 0.0002, Val Loss = 0.0001, Val MSE = 0.0001
  Init Acc: 0.0337, Final Acc: 0.0364, Tone Acc: 0.1631, Exact: 0.0002
  -> Best model saved (val loss: 0.0001)


Epoch 3/20 [Val]: 100%|██████████| 505/505 [01:05<00:00,  7.72it/s]


...

Epoch 18: Train Loss = 0.0000, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.0329, Final Acc: 0.0553, Tone Acc: 0.1831, Exact: 0.0000
  -> Best model saved (val loss: 0.0000)


Epoch 19/20 [Val]: 100%|██████████| 505/505 [01:05<00:00,  7.69it/s]


Epoch 19: Train Loss = 0.0000, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.0378, Final Acc: 0.0578, Tone Acc: 0.1823, Exact: 0.0000
  -> Best model saved (val loss: 0.0000)


Epoch 20/20 [Val]: 100%|██████████| 505/505 [01:05<00:00,  7.68it/s]


Epoch 20: Train Loss = 0.0000, Val Loss = 0.0000, Val MSE = 0.0000
  Init Acc: 0.0365, Final Acc: 0.0744, Tone Acc: 0.1804, Exact: 0.0000
  -> Best model saved (val loss: 0.0000)
Training complete.


## Testing

In [23]:
import torch
import torch.nn as nn
from tqdm import tqdm

def evaluate_bert_model(model, test_loader, device, num_initials, num_finals, num_tones, ignore_idx=-100):
    """
    Evaluate the BERT-based pinyin encoder on the test set.
    Returns a dictionary with initial, final, tone, and exact accuracies.
    """
    model.eval()
    correct_i = correct_f = correct_t = correct_exact = 0
    total_chars = 0

    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Evaluating on test set")
        for batch in test_pbar:
            input_ids, attention_mask, pinyin_comp, pinyin_idxs, char_lengths = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            pinyin_comp = pinyin_comp.to(device)           # (batch, max_seq_len, 3)
            # pinyin_idxs not used for accuracy, but can be used for MSE if needed

            logits_i, logits_f, logits_t = model(input_ids, attention_mask)

            pred_i = logits_i.argmax(dim=-1)   # (batch, max_seq_len)
            pred_f = logits_f.argmax(dim=-1)
            pred_t = logits_t.argmax(dim=-1)

            # Mask for valid character positions (non‑ignore)
            # The pinyin targets have ignore_idx at padding and (for the corrected dataset) also at special tokens
            mask = (pinyin_comp[..., 0] != ignore_idx)   # same for all components

            correct_i += ((pred_i == pinyin_comp[..., 0]) & mask).sum().item()
            correct_f += ((pred_f == pinyin_comp[..., 1]) & mask).sum().item()
            correct_t += ((pred_t == pinyin_comp[..., 2]) & mask).sum().item()
            exact = (pred_i == pinyin_comp[..., 0]) & (pred_f == pinyin_comp[..., 1]) & (pred_t == pinyin_comp[..., 2]) & mask
            correct_exact += exact.sum().item()
            total_chars += mask.sum().item()

    acc_i = correct_i / total_chars if total_chars > 0 else 0
    acc_f = correct_f / total_chars if total_chars > 0 else 0
    acc_t = correct_t / total_chars if total_chars > 0 else 0
    acc_exact = correct_exact / total_chars if total_chars > 0 else 0

    return {
        'initial_accuracy': acc_i,
        'final_accuracy': acc_f,
        'tone_accuracy': acc_t,
        'exact_accuracy': acc_exact
    }

In [24]:
# Load the saved state dict
model.load_state_dict(torch.load('bert_pinyin_best.pt', map_location=device))
model.to(device)

# Evaluate
test_metrics = evaluate_bert_model(model, test_loader, device, num_initials, num_finals, num_tones, ignore_idx=-100)
print("Test metrics:", test_metrics)

Evaluating on test set: 100%|██████████| 505/505 [01:08<00:00,  7.41it/s]

Test metrics: {'initial_accuracy': 0.03629805322695185, 'final_accuracy': 0.07499411562347233, 'tone_accuracy': 0.18126897334226552, 'exact_accuracy': 0.0}
